# Calibration under shift — results walkthrough

This notebook reads saved manifests and result tables; it never trains a model. The committed SMIDS/HuSHeM grid is the default scientific path. The synthetic demo is an explicitly opted-in CI fixture and is never interpreted as scientific evidence.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from experiments.analyze import validate_complete_grid
from experiments.checkpoint3 import build_main_table, validate_checkpoint3_inputs

ROOT = Path.cwd()
if not (ROOT / "configs").exists() and (ROOT.parent / "configs").exists():
    ROOT = ROOT.parent
def resolved_path(raw, default):
    candidate = Path(raw) if raw else Path(default)
    return (candidate if candidate.is_absolute() else ROOT / candidate).resolve()

ALLOW_DEMO = os.getenv("CALIBRATION_NOTEBOOK_ALLOW_DEMO") == "1"
METRICS_OVERRIDE = os.getenv("CALIBRATION_NOTEBOOK_METRICS")
THRESHOLDS_OVERRIDE = os.getenv("CALIBRATION_NOTEBOOK_THRESHOLDS")
if ALLOW_DEMO and (not METRICS_OVERRIDE or not THRESHOLDS_OVERRIDE):
    raise RuntimeError(
        "demo mode requires CALIBRATION_NOTEBOOK_METRICS and "
        "CALIBRATION_NOTEBOOK_THRESHOLDS"
    )
if not ALLOW_DEMO and (METRICS_OVERRIDE or THRESHOLDS_OVERRIDE):
    raise RuntimeError("fixture path overrides require CALIBRATION_NOTEBOOK_ALLOW_DEMO=1")
RESULTS = resolved_path(os.getenv("CALIBRATION_RESULTS_DIR"), ROOT / "results")
ATTRIBUTION = RESULTS / "attribution"
METRICS_PATH = resolved_path(METRICS_OVERRIDE, RESULTS / "metrics.csv")
THRESHOLDS_PATH = resolved_path(THRESHOLDS_OVERRIDE, RESULTS / "thresholds.csv")
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.2})
print(f"repository: {ROOT}")
print(f"results: {RESULTS}")

## 1. Prespecified question

The test is temporal in severity space: does a reliability signal cross its prespecified threshold before clean accuracy has fallen by five percentage points? Thresholds live in a versioned YAML file and are applied without substituting missing crossings.

In [ ]:
with (ROOT / "configs" / "analysis_protocol.yaml").open() as handle:
    protocol = yaml.safe_load(handle)
protocol

## 2. Data provenance and split contract

Training, validation, calibration, and test rows are fixed in manifests. Temperature scaling and APS conformal prediction use only the calibration role. Device corruptions are applied only at evaluation time.

In [ ]:
for dataset in ("smids", "hushem", "kromp"):
    path = ROOT / "data" / "splits" / f"{dataset}.csv"
    if path.exists():
        frame = pd.read_csv(path)
        index = ["fold", "split"] if "fold" in frame else ["split"]
        print(f"\n{dataset}: {len(frame):,} manifest rows")
        display(frame.groupby(index).size().rename("rows").to_frame())
    else:
        print(f"\n{dataset}: no manifest (see results/data_audit.md)")

## 3. Load completed result rows

By default, only the committed canonical `results/metrics.csv` is treated as scientific output. Set `CALIBRATION_RESULTS_DIR` for an identically structured reproduction tree. Demo execution requires `CALIBRATION_NOTEBOOK_ALLOW_DEMO=1` plus explicit metrics and threshold paths.

In [ ]:
required = {"dataset", "model", "seed", "fold", "corruption", "severity", "method", "metric", "value"}
if not METRICS_PATH.is_file():
    raise FileNotFoundError(f"result table not found: {METRICS_PATH}")
metrics = pd.read_csv(METRICS_PATH)
missing = required - set(metrics)
if missing:
    raise ValueError(f"metrics file is missing columns: {sorted(missing)}")
datasets = set(metrics["dataset"].astype(str))
demo_only = datasets == {"synthetic_demo"}
if demo_only:
    if not ALLOW_DEMO:
        raise ValueError("synthetic demo rows require explicit demo opt-in")
    print("CI-only synthetic fixture: exercising result cells; not scientific output.")
else:
    if ALLOW_DEMO or "synthetic_demo" in datasets:
        raise ValueError("demo mode and scientific results cannot be mixed")
    validate_complete_grid(metrics, protocol)
    print("validated complete prespecified SMIDS/HuSHeM grid")
print(f"{len(metrics):,} tidy rows from {METRICS_PATH}")

## 4. Clean baselines

Accuracy and macro-F1 are summarized across independent seeds (or HuSHeM folds). This is a quick integrity check, not the headline shift analysis.

In [ ]:
if metrics.empty:
    print("Skipped: no public-data metrics.")
else:
    clean = metrics.query("corruption == 'clean' and method == 'raw_softmax'")
    baseline = (clean[clean.metric.isin(["accuracy", "macro_f1", "ece", "nll"])]
                .groupby(["dataset", "model", "metric"]).value
                .agg(["mean", "std", "count"]))
    display(baseline)

## 5. Primary/prespecified trajectories

Corruptions are first averaged equally within each seed, then seeds are summarized. This prevents a corruption with more rows from receiving extra weight.

In [ ]:
def device_summary(metric_name, method):
    names = set(protocol["aggregation"]["device_corruptions"])
    selected = metrics[(metrics.corruption.isin(names)) &
                       (metrics.metric == metric_name) &
                       (metrics.method == method)].copy()
    if selected.empty:
        return pd.DataFrame()
    within = (selected.groupby(["dataset", "model", "seed", "fold", "severity"], as_index=False, dropna=False)
              .value.mean())
    return (within.groupby(["dataset", "model", "severity"]).value
            .agg(["mean", "std", "count"]).reset_index())

accuracy = device_summary("accuracy", "raw_softmax")
ece = device_summary("ece", "raw_softmax")
if accuracy.empty:
    print("Skipped: no complete public-data grid.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for keys, group in accuracy.groupby(["dataset", "model"]):
        axes[0].plot(group.severity, group["mean"], marker="o", label="/".join(keys))
    for keys, group in ece.groupby(["dataset", "model"]):
        axes[1].plot(group.severity, group["mean"], marker="o", label="/".join(keys))
    axes[0].set(xlabel="Severity", ylabel="Accuracy", title="Accuracy")
    axes[1].set(xlabel="Severity", ylabel="ECE", title="Calibration error")
    axes[0].legend(frameon=False)
    plt.tight_layout()

## 6. Secondary/exploratory: temperature transfer

Temperature is optimized once on clean calibration logits. Comparing raw and temperature-scaled ECE under corruption asks whether that clean post-hoc correction transfers; it is not refit on the shifted test set.

In [ ]:
if metrics.empty:
    print("Skipped: no public-data metrics.")
else:
    names = set(protocol["aggregation"]["device_corruptions"])
    comparison = metrics[(metrics.metric == "ece") &
                         (metrics.method.isin(["raw_softmax", "temperature"])) &
                         (metrics.corruption.isin(names))]
    within = (comparison.groupby(
        ["dataset", "model", "method", "seed", "fold", "severity"],
        as_index=False, dropna=False).value.mean())
    display(within.groupby(["dataset", "model", "method", "severity"]).value
            .agg(["mean", "std", "count"]).head(30))

## 7. Secondary/exploratory: selective prediction and conformal sets

Risk at 80% coverage asks whether abstention can retain safer cases. APS reports empirical coverage and mean set size; its nominal guarantee relies on exchangeability and can fail under shift.

In [ ]:
if metrics.empty:
    print("Skipped: no public-data metrics.")
else:
    names = set(protocol["aggregation"]["device_corruptions"])
    selected = metrics[metrics.metric.isin([
        "risk_at_80_coverage", "conformal_coverage", "conformal_mean_set_size"
    ]) & metrics.corruption.isin(names)]
    within = (selected.groupby(
        ["dataset", "model", "method", "metric", "seed", "fold", "severity"],
        as_index=False, dropna=False).value.mean())
    display(within.groupby(
        ["dataset", "model", "method", "metric", "severity"]
    ).value.agg(["mean", "std", "count"]).head(40))

## 8. Prespecified threshold result

The generated table is the direct answer to the prespecified question. An early-warning gap is positive only when a reliability threshold crosses at a lower severity than the five-point accuracy-loss threshold. Missing crossings stay missing.

In [ ]:
if not THRESHOLDS_PATH.is_file():
    raise FileNotFoundError(f"threshold table not found: {THRESHOLDS_PATH}")
thresholds = pd.read_csv(THRESHOLDS_PATH)
if demo_only:
    print("Synthetic threshold fixture only; no scientific conclusion is drawn.")
    display(thresholds)
else:
    metrics, thresholds = validate_checkpoint3_inputs(metrics, thresholds, protocol)
    main_table = build_main_table(metrics, thresholds)
    status_counts = main_table["status"].value_counts().to_dict()
    observed = int(status_counts.get("earlier", 0) + status_counts.get("same_or_later", 0))
    earlier = int(status_counts.get("earlier", 0))
    never_crossed = int(status_counts.get("signal_did_not_cross", 0))
    assert (observed, earlier, never_crossed) == (10, 0, 6)
    print(
        "Primary/prespecified result: no reliability signal crossed before the "
        f"accuracy-drop threshold in any of the {observed} comparisons with both "
        f"crossings observed; {never_crossed} further signals never crossed. "
        "The early-warning hypothesis was not supported at these thresholds on "
        "SMIDS and HuSHeM."
    )
    display(main_table[[
        "analysis_tier", "dataset", "model", "signal_label",
        "signal_crossing_severity", "accuracy_drop_severity", "status"
    ]])

## 9. Secondary/exploratory: attribution stability

Grad-CAM comparisons use fixed clean targets and paired images. Spearman agreement and top-20% saliency IoU quantify drift; pictures alone are not treated as evidence.

In [ ]:
stability_path = ATTRIBUTION / "attribution_stability.csv"
if demo_only:
    print("Skipped for the synthetic fixture; attribution is not scientific output.")
elif stability_path.exists():
    stability = pd.read_csv(stability_path)
    display(stability.groupby("severity")[["spearman", "top_percent_iou"]]
            .agg(["mean", "std", "count"]))
else:
    raise FileNotFoundError(f"real-checkpoint attribution table not found: {stability_path}")

## 10. Interpretation discipline

The bounded null applies to the frozen threshold-level early-warning claim on these datasets; it is not proof that uncertainty methods are useless. Classification degradation, miscalibration, per-sample failure ranking, conformal coverage under broken exchangeability, input-shift detection, and attribution drift answer different operational questions. All analyses outside the frozen threshold decision above are secondary/exploratory.

## 11. Limitations

SMIDS and HuSHeM are public proxy datasets with no released donor mapping; Kromp cannot currently be split by patient from its public metadata. Corruptions are sensitivity analyses, not paired smartphone/clinical captures. The study makes no clinical-performance or deployment claim.

## 12. Primary references

- [Thirumalaraju et al., Fertility and Sterility (online 2025; issue 2026)](https://doi.org/10.1016/j.fertnstert.2025.08.021)
- [Kanakasabapathy et al., Nature Biomedical Engineering (2021)](https://doi.org/10.1038/s41551-021-00733-w)
- [Guo et al., ICML (2017)](https://proceedings.mlr.press/v70/guo17a.html)
- [Ovadia et al., NeurIPS (2019)](https://proceedings.neurips.cc/paper_files/paper/2019/hash/8558cb408c1d76621371888657d2eb1d-Abstract.html)
- [Angelopoulos & Bates (2023)](https://doi.org/10.1561/2200000101)

## 13. Next experiment

The decisive extension is a paired acquisition study: image the same specimen through a reference microscope and the lab's smartphone hardware, then compare this synthetic severity ordering with real device-domain metrics while keeping patients grouped and calibration data isolated.